In [17]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import glob
import re

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 35
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## IMF Fiscal Rules Pipeline

**Source:** IMF Fiscal Affairs Department — Fiscal Rules Dataset
**Access:** Manual Excel download (auto-detects file in Downloads)
**Download instructions:** See `docs/instructions_data_maintenance.md` — IMF_FISCAL_RULES section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Presence of fiscal rules (budget balance, debt, expenditure, revenue) | Macroeconomic policy framework quality | Primary tier 1 |

In [3]:
import pandas as pd
import os
import glob
from datetime import datetime

# Auto-detect IMF Fiscal Rules Excel in Downloads — no hardcoded filename
fr_pattern = os.path.join(DOWNLOADS_DIR, "*Fiscal Rules*.xlsx")
fr_files = glob.glob(fr_pattern)

if not fr_files:
    print(f"No IMF Fiscal Rules file found in {DOWNLOADS_DIR}")
    print("Download from imf.org/en/topics/fiscal-policies/fiscal-rules-dataset")
else:
    # Use most recent download
    fr_file = max(fr_files, key=os.path.getmtime)
    print(f"Found: {os.path.basename(fr_file)}")
    
    # Inspect sheets
    xl = pd.ExcelFile(fr_file, engine='openpyxl')
    print(f"Sheets: {xl.sheet_names}")

Found: Publication - IMF FAD Fiscal Rules Dataset 1985-2024 Update.xlsx
Sheets: ['Cover', 'Abbreviation and codes', 'Revisions NEW', 'Rules', 'Supranational', 'European Union', 'EU ', 'ECCU', 'EAMU (EAC)', 'CEMAC', 'WEAMU', 'FAD own checking', 'WEO_IFS', 'weo_group']


In [4]:
# Inspect the Rules sheet — the main national fiscal rules data
# Try reading with no header first to see the layout
rules_preview = pd.read_excel(fr_file, sheet_name='Rules', engine='openpyxl', header=None, nrows=5)
print("First 5 rows (no header):")
print(rules_preview.to_string())
print(f"\nFull sheet shape: {pd.read_excel(fr_file, sheet_name='Rules', engine='openpyxl', header=None).shape}")

First 5 rows (no header):
         0     1             2                             3                 4                         5              6                                          7                 8                         9              10                      11                12                        13             14                                                       15                16                        17             18                                                      19         20                 21                                                                                22                                           23   24   25   26                   27   28   29   30                             31   32   33   34                   35   36   37   38              39   40   41   42                   43   44   45   46                                       47   48   49   50                   51   52   53   54              55   56   57   58                 

In [13]:
# Load Rules sheet using robust NAME-BASED column selection (not positions)
# Composite keys built from nested header: parent rows 0-2 forward-filled, leaf row 3 un-filled
# This survives column reordering; fails loudly if IMF renames a header section
rules_raw_full = pd.read_excel(fr_file, sheet_name='Rules', engine='openpyxl', header=None)

# Rebuild composite keys: forward-fill parent rows only, leave leaf row un-filled
_hdr = rules_raw_full.iloc[0:4]
_filled_top = _hdr.iloc[0:3].ffill(axis=1)
_leaf = _hdr.iloc[3]

def _composite_key(col_idx):
    parts = [str(_filled_top.iloc[r, col_idx]).strip() for r in range(3)
             if pd.notna(_filled_top.iloc[r, col_idx]) and str(_filled_top.iloc[r, col_idx]).strip() != 'nan']
    leaf = _leaf.iloc[col_idx]
    if pd.notna(leaf) and str(leaf).strip() != 'nan':
        parts.append(str(leaf).strip())
    return ' || '.join(parts)

_keys = {i: _composite_key(i) for i in range(rules_raw_full.shape[1])}

def find_col(*must_contain):
    """Return the single column index whose composite key contains all given fragments.
    Raises if zero or multiple matches — surfaces template changes loudly."""
    matches = [i for i, k in _keys.items() if all(frag in k for frag in must_contain)]
    if len(matches) != 1:
        raise ValueError(f"Expected 1 column matching {must_contain}, found {len(matches)}: "
                         f"{[_keys[i] for i in matches]}")
    return matches[0]

# Map clean output names to the header fragments that uniquely identify each column.
# 'exact' = full composite key must match exactly (for short identifier keys).
# Otherwise = all fragments must appear as substrings in the composite key.
col_spec = {
    'year':                                ('exact', 'year'),
    'country_name':                        ('exact', 'Country name'),
    # Presence per rule type
    'fr_expenditure_rule':                 ('Type of fiscal rule in place', 'ER'),
    'fr_revenue_rule':                     ('Type of fiscal rule in place', 'RR'),
    'fr_budget_balance_rule':              ('Type of fiscal rule in place', 'BBR'),
    'fr_debt_rule':                        ('Type of fiscal rule in place', 'DR'),
    # Legal basis per rule type (national) — quality signal
    'fr_legal_basis_er':                   ('Legal or political basis', 'ER', 'National'),
    'fr_legal_basis_rr':                   ('Legal or political basis', 'RR', 'National'),
    'fr_legal_basis_bbr':                  ('Legal or political basis', 'BBR', 'National'),
    'fr_legal_basis_dr':                   ('Legal or political basis', 'DR', 'National'),
    # Formal enforcement procedure per rule type (national)
    'fr_enforcement_er':                   ('Formal enforcement procedure', 'National', 'ER'),
    'fr_enforcement_rr':                   ('Formal enforcement procedure', 'National', 'RR'),
    'fr_enforcement_bbr':                  ('Formal enforcement procedure', 'National', 'BBR'),
    'fr_enforcement_dr':                   ('Formal enforcement procedure', 'National', 'DR'),
    # Independent monitoring institutions
    'fr_indep_body_sets_assumptions':      ('Independent body sets budget assumptions',),
    'fr_indep_body_monitors':              ('Independent body monitors implementation',),
    # Correction mechanism
    'fr_correction_mechanism':             ('Presence of correction mechanism',),
    'fr_correction_well_defined_triggers': ('prespecifies well-defined triggers',),
    # Compliance per rule type (national)
    'fr_compliance_er':                    ('Does the country comply', 'ER'),
    'fr_compliance_rr':                    ('Does the country comply', 'RR'),
    'fr_compliance_bbr':                   ('Does the country comply', 'BBR'),
    'fr_compliance_dr':                    ('Does the country comply', 'DR'),
    # Identifiers
    'ifs_code':                            ('ifscode',),
    'country_code':                        ('ccode',),
}

# Resolve each spec to a column index
# 'exact' specs match the full composite key exactly; others match by substring fragments
resolved = {}
for out_name, frags in col_spec.items():
    if frags[0] == 'exact':
        matches = [i for i, k in _keys.items() if k == frags[1]]
        if len(matches) != 1:
            raise ValueError(f"Expected 1 exact match for '{frags[1]}', found {len(matches)}")
        resolved[out_name] = matches[0]
    else:
        resolved[out_name] = find_col(*frags)

# Data starts at row 4 (after the 4 header rows)
fr = rules_raw_full.iloc[4:, list(resolved.values())].copy()
fr.columns = list(resolved.keys())

print(f"Resolved {len(resolved)} columns by name")
print(f"Raw shape: {fr.shape}")
print(fr.head(3).to_string())

Resolved 24 columns by name
Raw shape: (4920, 24)
   year country_name fr_expenditure_rule fr_revenue_rule fr_budget_balance_rule fr_debt_rule fr_legal_basis_er fr_legal_basis_rr fr_legal_basis_bbr fr_legal_basis_dr fr_enforcement_er fr_enforcement_rr fr_enforcement_bbr fr_enforcement_dr fr_indep_body_sets_assumptions fr_indep_body_monitors fr_correction_mechanism fr_correction_well_defined_triggers fr_compliance_er fr_compliance_rr fr_compliance_bbr fr_compliance_dr ifs_code country_code
4  1985      Andorra                   -               -                      -            -                 -                 -                  -                 -                 -                 -                  -                 -                              -                      -                       -                                   -                -                -                 -                -      171          AND
5  1986      Andorra                   -               -      

In [18]:
# Normalize quality dimensions into clean numeric measures
# Legal basis is an ordinal strength scale per IMF codebook:
#   1=Political commitment, 2=Coalition agreement, 3=Statutory, 4=Int'l Treaty, 5=Constitutional
# Higher = stronger legal entrenchment of the rule

import numpy as np

presence_cols = ['fr_expenditure_rule', 'fr_revenue_rule', 'fr_budget_balance_rule', 'fr_debt_rule']
legal_cols    = ['fr_legal_basis_er', 'fr_legal_basis_rr', 'fr_legal_basis_bbr', 'fr_legal_basis_dr']
enforce_cols  = ['fr_enforcement_er', 'fr_enforcement_rr', 'fr_enforcement_bbr', 'fr_enforcement_dr']
comply_cols   = ['fr_compliance_er', 'fr_compliance_rr', 'fr_compliance_bbr', 'fr_compliance_dr']

def to_num(series):
    """Coerce a column to numeric: '-' and 'n.a.' become NaN; messy multi-codes like '1,2' take the max."""
    def parse(v):
        s = str(v).strip()
        if s in ('-', 'n.a.', 'nan', ''):
            return np.nan
        # Handle messy multi-value entries like "1,2" or "1 and 2" — take the strongest (max)
        nums = [int(x) for x in re.findall(r'\d+', s)]
        return max(nums) if nums else np.nan
    return series.map(parse)

# Presence → binary (1 if "1", else 0)
for col in presence_cols:
    fr[col] = (fr[col].astype(str).str.strip() == '1').astype(int)

# Legal basis → ordinal 1-5 (NaN where no rule)
for col in legal_cols:
    fr[col] = to_num(fr[col])

# Enforcement, independent monitoring, correction, triggers → numeric
for col in enforce_cols + ['fr_indep_body_sets_assumptions', 'fr_indep_body_monitors',
                            'fr_correction_mechanism', 'fr_correction_well_defined_triggers']:
    fr[col] = to_num(fr[col])

# Compliance → numeric (1=complies, 0=does not; 2 is a rare code, keep as-is; n.a.→NaN)
for col in comply_cols:
    fr[col] = to_num(fr[col])

# Count of rule types in place
fr['fr_num_rule_types'] = fr[presence_cols].sum(axis=1)

# Aggregate strength signal: highest legal basis across any rule type in place (NaN if no rules)
fr['fr_max_legal_basis'] = fr[legal_cols].max(axis=1)

# Mean legal basis across rule types that exist (ignores NaN)
fr['fr_mean_legal_basis'] = fr[legal_cols].mean(axis=1)

# Any formal enforcement across rule types
fr['fr_any_enforcement'] = (fr[enforce_cols].fillna(0).sum(axis=1) > 0).astype(int)

# Year to numeric, filter to framework start
fr['year'] = pd.to_numeric(fr['year'], errors='coerce').astype('Int64')
fr = fr[fr['year'].notna() & fr['country_name'].notna()].copy()
fr = fr[fr['year'] >= FRAMEWORK_START_YEAR].copy()
fr = fr.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {fr.shape}")
print(f"Years: {fr['year'].min()} — {fr['year'].max()}")
print(f"Countries: {fr['country_name'].nunique()}")
print(f"\nMax legal basis distribution (country-years with any rule):")
print(fr[fr['fr_num_rule_types'] > 0]['fr_max_legal_basis'].value_counts().sort_index())
print(f"\nSample (recent years, a country with rules):")
print(fr[fr['fr_num_rule_types'] > 0][['country_name','year','fr_num_rule_types','fr_max_legal_basis','fr_any_enforcement','fr_indep_body_monitors']].tail(5).to_string())

Shape: (4305, 28)
Years: 1990 — 2024
Countries: 123

Max legal basis distribution (country-years with any rule):
fr_max_legal_basis
1.0     178
2.0     204
3.0    1149
5.0     260
Name: count, dtype: int64

Sample (recent years, a country with rules):
     country_name  year  fr_num_rule_types  fr_max_legal_basis  fr_any_enforcement  fr_indep_body_monitors
4268      Vietnam  2023                  3                 3.0                   0                     0.0
4269      Vietnam  2024                  3                 3.0                   0                     0.0
4302       Zambia  2022                  1                 3.0                   0                     1.0
4303       Zambia  2023                  1                 3.0                   0                     1.0
4304       Zambia  2024                  1                 3.0                   0                     1.0


In [19]:
# Derive metadata from data — no hardcoding
latest_year = str(int(fr['year'].max()))

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "imf_fiscal_rules_clean.csv")
fr.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {fr.shape}")

# Update download log
update_entry(
    "IMF_FISCAL_RULES",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="imf_fiscal_rules_clean.csv",
    latest_available_version=latest_year,
    notes="Fiscal rule presence AND quality. Per rule type (ER/RR/BBR/DR): presence, legal basis "
          "(1=political commitment to 5=constitutional), formal enforcement, compliance. Plus: independent "
          "monitoring body, correction mechanism, well-defined triggers. Derived: count of rule types, "
          "max/mean legal basis, any enforcement. Robust name-based column selection from nested header — "
          "survives column reordering, fails loudly on section rename. Coverage: 123 countries."
)
print_entry("IMF_FISCAL_RULES")

Written: C:\Users\mjbou\governance-framework\data\processed\imf_fiscal_rules_clean.csv
Shape: (4305, 28)
[download_log] Updated entry for IMF_FISCAL_RULES
  source_id: IMF_FISCAL_RULES
  last_attempted_date: 2026-06-17
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2024
  local_filename: imf_fiscal_rules_clean.csv
  latest_available_version: 2024
  no_update_reason: nan
  notes: Fiscal rule presence AND quality. Per rule type (ER/RR/BBR/DR): presence, legal basis (1=political commitment to 5=constitutional), formal enforcement, compliance. Plus: independent monitoring body, correction mechanism, well-defined triggers. Derived: count of rule types, max/mean legal basis, any enforcement. Robust name-based column selection from nested header — survives column reordering, fails loudly on section rename. Coverage: 123 countries.
